In [6]:
import os
import sys
import json
import hashlib
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch

# =====================================================================
# 1. DYNAMIC PROJECT ROOT & DIRECTORY AUTO-DETECTION
# =====================================================================
CURRENT_DIR = Path.cwd().resolve()

# Auto-detect BTech_Project root directory
if (CURRENT_DIR / "datasets").exists():
    BTECH_ROOT = CURRENT_DIR
elif (CURRENT_DIR / "BTech_Project" / "datasets").exists():
    BTECH_ROOT = CURRENT_DIR / "BTech_Project"
elif (CURRENT_DIR.parent / "datasets").exists():
    BTECH_ROOT = CURRENT_DIR.parent
elif (CURRENT_DIR.parent.parent / "datasets").exists():
    BTECH_ROOT = CURRENT_DIR.parent.parent
else:
    raise FileNotFoundError("Could not auto-locate 'BTech_Project/datasets' folder.")

DATASETS_DIR = BTECH_ROOT / "datasets"
CLINICAL_DIR = DATASETS_DIR / "clinical_multisource"
DERMNET_DIR = DATASETS_DIR / "dermnet_original"
METADATA_DIR = BTECH_ROOT  / "metadata"
DATASET_META_DIR = DATASETS_DIR / "metadata"

# Ensure output metadata directory exists
METADATA_DIR.mkdir(parents=True, exist_ok=True)

# =====================================================================
# 2. INTEGRITY VERIFICATION
# =====================================================================
print("=" * 70)
print("  PHASE 1: MULTI-SOURCE DATASET HARMONIZATION & INGESTION")
print("=" * 70)
print(f"✓ Project Root        : {BTECH_ROOT}")
print(f"✓ Datasets Directory  : {DATASETS_DIR}")
print(f"✓ Clinical MultiSource: {CLINICAL_DIR} (Exists: {CLINICAL_DIR.exists()})")
print(f"✓ DERMNet Original    : {DERMNET_DIR} (Exists: {DERMNET_DIR.exists()})")
print(f"✓ Target Metadata Dir : {METADATA_DIR}")

# Check clinical manifest
manifest_path = DATASET_META_DIR / "manifest.csv"
if not manifest_path.exists():
    # Fallback check
    manifest_path = METADATA_DIR / "manifest.csv"

print(f"✓ Clinical Manifest   : {manifest_path} (Exists: {manifest_path.exists()})")

# =====================================================================
# 3. COMPUTE HARDWARE VERIFICATION
# =====================================================================
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("-" * 70)
print(f"✓ Active Device       : {DEVICE}")
if DEVICE.type == "cuda":
    print(f"✓ GPU Accelerator     : {torch.cuda.get_device_name(0)}")
    print(f"✓ VRAM Available      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
print("=" * 70)
print("Environment initialized successfully. Ready for dataset scanning.")


  PHASE 1: MULTI-SOURCE DATASET HARMONIZATION & INGESTION
✓ Project Root        : C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project
✓ Datasets Directory  : C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project\datasets
✓ Clinical MultiSource: C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project\datasets\clinical_multisource (Exists: True)
✓ DERMNet Original    : C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project\datasets\dermnet_original (Exists: True)
✓ Target Metadata Dir : C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project\metadata
✓ Clinical Manifest   : C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project\datasets\metadata\manifest.csv (Exists: True)
----------------------------------------------------------------------
✓ Active Devi

In [2]:
# =====================================================================
# 1. UNIFIED CLINICAL TAXONOMY DEFINITION (DUAL-TIER HIERARCHY)
# =====================================================================
# Tier 1 = Core Motive: Systemic Manifestations, NTDs, High-Risk Malignancy
# Tier 2 = Broad Spectrum: Routine High-Frequency Primary-Care Dermatology

UNIFIED_TAXONOMY = {
    # --- TIER 1: CORE MOTIVE & HIGH-PRIORITY CLINICAL TARGETS ---
    "Malignant_Melanoma": {
        "tier": "Tier 1",
        "urgency": "High-Priority Oncology",
        "category": "Neoplastic Malignancy",
        "description": "Invasive cutaneous melanoma and malignant pigmented lesions"
    },
    "Kaposi_Sarcoma": {
        "tier": "Tier 1",
        "urgency": "High-Priority Systemic",
        "category": "HIV-Associated Vascular Neoplasm",
        "description": "HHV-8 vascular neoplasm presenting in immunocompromised patients"
    },
    "Scabies_and_Infestations": {
        "tier": "Tier 1",
        "urgency": "Urgent Public Health",
        "category": "Neglected Tropical Disease (NTD)",
        "description": "Sarcoptes scabiei burrows and neglected tropical ectoparasitic eruptions"
    },
    "Systemic_Lupus_Erythematosus": {
        "tier": "Tier 1",
        "urgency": "Urgent Systemic Consultation",
        "category": "Autoimmune Connective Tissue",
        "description": "Malar butterfly erythema, subacute cutaneous, and discoid lupus"
    },
    "Cutaneous_Vasculitis": {
        "tier": "Tier 1",
        "urgency": "Urgent Systemic Consultation",
        "category": "Systemic Vascular Inflammation",
        "description": "Leukocytoclastic vasculitis and palpable purpuric eruptions"
    },
    "Bullous_Pemphigoid": {
        "tier": "Tier 1",
        "urgency": "Urgent Dermatology Referral",
        "category": "Severe Autoimmune Blistering",
        "description": "Subepidermal tense blistering, pemphigus, and bullous dermatoses"
    },
    "Cellulitis": {
        "tier": "Tier 1",
        "urgency": "Emergency Infection Protocol",
        "category": "Deep Bacterial Infection",
        "description": "Deep dermal acute spreading erythema, edema, and warmth"
    },
    "Exanthems_and_Drug_Eruptions": {
        "tier": "Tier 1",
        "urgency": "Urgent Toxicology/Allergy",
        "category": "Adverse Cutaneous Reactions",
        "description": "Severe cutaneous adverse reactions (SCARs) and morbilliform drug eruptions"
    },
    "Systemic_Disease_Manifestations": {
        "tier": "Tier 1",
        "urgency": "Urgent Internal Medicine",
        "category": "Cutaneous Marker of Systemic Disease",
        "description": "Cutaneous markers of renal, hepatic, endocrine, and metabolic disorders"
    },

    # --- TIER 2: BROAD GENERAL PRIMARY CARE DERMATOLOGY ---
    "Melanocytic_Nevi": {
        "tier": "Tier 2",
        "urgency": "Routine Screening",
        "category": "Benign Melanocytic Lesion",
        "description": "Benign junctional, compound, dermal moles and congenital nevi"
    },
    "Seborrheic_Keratosis": {
        "tier": "Tier 2",
        "urgency": "Routine Care",
        "category": "Benign Keratinocyte Tumor",
        "description": "Benign warty, hyperkeratotic 'stuck-on' epidermal proliferations"
    },
    "Acne_Vulgaris": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Pilosebaceous Inflammatory",
        "description": "Comedonal, inflammatory papular, and pustular acne"
    },
    "Rosacea": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Facial Vascular/Erythematous",
        "description": "Centrofacial erythema, telangiectasia, and papulopustular rosacea"
    },
    "Atopic_Dermatitis_and_Eczema": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Pruritic Eczematous",
        "description": "Flexural atopic eczema, acute/subacute spongiotic dermatitis"
    },
    "Contact_Dermatitis": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Allergic & Irritant Contact",
        "description": "Poison ivy, plant dermatitis, and occupational allergic eruptions"
    },
    "Psoriasis_and_Lichen_Planus": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Papulosquamous / Plaque",
        "description": "Silvery micaceous scale plaques and violaceous lichenoid papules"
    },
    "Onychomycosis_and_Nail_Disease": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Nail Apparatus Fungal",
        "description": "Subungual hyperkeratosis, tinea unguium, and fungal dystrophy"
    },
    "Tinea_and_Fungal_Infections": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Cutaneous Dermatophytosis",
        "description": "Tinea corporis (ringworm), cruris, pedis, and cutaneous candidiasis"
    },
    "Impetigo_and_Bacterial": {
        "tier": "Tier 2",
        "urgency": "Prompt Outpatient Treatment",
        "category": "Superficial Bacterial",
        "description": "Honey-colored crusted contagious bacterial skin lesions"
    },
    "Viral_Warts_and_Molluscum": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Viral Epithelial Eruption",
        "description": "Verruca vulgaris, plantar warts, and molluscum contagiosum"
    },
    "Herpes_HPV_and_STDs": {
        "tier": "Tier 2",
        "urgency": "Prompt Outpatient Treatment",
        "category": "Viral & Venereal Dermatoses",
        "description": "Herpes simplex, herpes zoster, and condylomata acuminata"
    },
    "Chronic_Urticaria": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Vascular Edematous / Wheals",
        "description": "Pruritic edematous transient hives and dermographism"
    },
    "Alopecia_Areata_and_Hair_Loss": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Follicular Hair Pathology",
        "description": "Patchy autoimmune hair loss and telogen effluvium"
    },
    "Vitiligo_and_Pigmentation": {
        "tier": "Tier 2",
        "urgency": "Routine Outpatient",
        "category": "Melanocyte Depigmentation",
        "description": "Autoimmune depigmented macules, melasma, and hyperpigmentation"
    },
    "Vascular_Tumors": {
        "tier": "Tier 2",
        "urgency": "Routine Care",
        "category": "Vascular Proliferation",
        "description": "Cherry angiomas, pyogenic granulomas, and benign hemangiomas"
    }
}

# Generate numeric label encoding
CLASS_NAMES = sorted(list(UNIFIED_TAXONOMY.keys()))
LABEL_TO_ID = {name: idx for idx, name in enumerate(CLASS_NAMES)}
ID_TO_LABEL = {idx: name for idx, name in enumerate(CLASS_NAMES)}

# Save standard label map
with open(METADATA_DIR / "unified_label_map.json", "w", encoding="utf-8") as f:
    json.dump(LABEL_TO_ID, f, indent=4)

# =====================================================================
# 2. SOURCE-SPECIFIC HARMONIZATION MAPPERS
# =====================================================================
# A. Mapping manifest.csv classes -> Unified Classes
MANIFEST_MAPPING = {
    "Acne_Vulgaris": "Acne_Vulgaris",
    "Rosacea": "Rosacea",
    "Atopic_Dermatitis": "Atopic_Dermatitis_and_Eczema",
    "Seborrheic_Dermatitis": "Atopic_Dermatitis_and_Eczema",
    "Contact_Dermatitis": "Contact_Dermatitis",
    "Bullous_Pemphigoid": "Bullous_Pemphigoid",
    "Cellulitis": "Cellulitis",
    "Impetigo": "Impetigo_and_Bacterial",
    "Alopecia_Areata": "Alopecia_Areata_and_Hair_Loss",
    "Malignant_Melanoma": "Malignant_Melanoma",
    "Melanocytic_Nevi": "Melanocytic_Nevi",
    "Onychomycosis": "Onychomycosis_and_Nail_Disease",
    "Psoriasis": "Psoriasis_and_Lichen_Planus",
    "Scabies": "Scabies_and_Infestations",
    "Seborrheic_Keratosis": "Seborrheic_Keratosis",
    "Systemic_Lupus_Erythematosus": "Systemic_Lupus_Erythematosus",
    "Cutaneous_Vasculitis": "Cutaneous_Vasculitis",
    "Chronic_Urticaria": "Chronic_Urticaria",
    "Vascular_Tumors": "Vascular_Tumors",
    "Viral_Warts": "Viral_Warts_and_Molluscum",
    "Vitiligo": "Vitiligo_and_Pigmentation",
    "Kaposi_Sarcoma": "Kaposi_Sarcoma"
}

# B. Mapping DERMNet Original folder names -> Unified Classes
DERMNET_FOLDER_MAPPING = {
    "Acne and Rosacea Photos": "Acne_Vulgaris",  # Will use filename disambiguation below
    "Actinic Keratosis Basal Cell Carcinoma and other Malignant Lesions": "Malignant_Melanoma",
    "Atopic Dermatitis Photos": "Atopic_Dermatitis_and_Eczema",
    "Bullous Disease Photos": "Bullous_Pemphigoid",
    "Cellulitis Impetigo and other Bacterial Infections": "Cellulitis",
    "Eczema Photos": "Atopic_Dermatitis_and_Eczema",
    "Exanthems and Drug Eruptions": "Exanthems_and_Drug_Eruptions",
    "Hair Loss Photos Alopecia and other Hair Diseases": "Alopecia_Areata_and_Hair_Loss",
    "Herpes HPV and other STDs Photos": "Herpes_HPV_and_STDs",
    "Light Diseases and Disorders of Pigmentation": "Vitiligo_and_Pigmentation",
    "Lupus and other Connective Tissue diseases": "Systemic_Lupus_Erythematosus",
    "Melanoma Skin Cancer Nevi and Moles": "Melanocytic_Nevi",  # Will use filename disambiguation below
    "Nail Fungus and other Nail Disease": "Onychomycosis_and_Nail_Disease",
    "Poison Ivy Photos and other Contact Dermatitis": "Contact_Dermatitis",
    "Psoriasis pictures Lichen Planus and related diseases": "Psoriasis_and_Lichen_Planus",
    "Scabies Lyme Disease and other Infestations and Bites": "Scabies_and_Infestations",
    "Seborrheic Keratoses and other Benign Tumors": "Seborrheic_Keratosis",
    "Systemic Disease": "Systemic_Disease_Manifestations",
    "Tinea Ringworm Candidiasis and other Fungal Infections": "Tinea_and_Fungal_Infections",
    "Urticaria Hives": "Chronic_Urticaria",
    "Vascular Tumors": "Vascular_Tumors",
    "Vasculitis Photos": "Cutaneous_Vasculitis",
    "Warts Molluscum and other Viral Infections": "Viral_Warts_and_Molluscum"
}

def disambiguate_dermnet_class(folder_name, filename):
    """Refines compound DERMNet folders into exact clinical diagnoses based on filename stems."""
    fname_lower = filename.lower()
    
    if folder_name == "Acne and Rosacea Photos":
        if "rosacea" in fname_lower:
            return "Rosacea"
        return "Acne_Vulgaris"
        
    elif folder_name == "Melanoma Skin Cancer Nevi and Moles":
        if "melanoma" in fname_lower or "malignant" in fname_lower:
            return "Malignant_Melanoma"
        return "Melanocytic_Nevi"
        
    elif folder_name == "Cellulitis Impetigo and other Bacterial Infections":
        if "impetigo" in fname_lower:
            return "Impetigo_and_Bacterial"
        return "Cellulitis"
        
    return DERMNET_FOLDER_MAPPING.get(folder_name, None)

# Print Summary
tier1_count = sum(1 for v in UNIFIED_TAXONOMY.values() if v['tier'] == 'Tier 1')
tier2_count = sum(1 for v in UNIFIED_TAXONOMY.values() if v['tier'] == 'Tier 2')

print("=" * 70)
print("  UNIFIED CLINICAL TAXONOMY ESTABLISHED")
print("=" * 70)
print(f"✓ Total Standardized Categories : {len(CLASS_NAMES)}")
print(f"✓ Tier 1 (Core Motive / NTDs)   : {tier1_count} categories (High Clinical Urgency)")
print(f"✓ Tier 2 (Broad Primary Care)   : {tier2_count} categories (Routine Frontline Triage)")
print(f"✓ Master Label Map Written      : {METADATA_DIR / 'unified_label_map.json'}")
print("=" * 70)

  UNIFIED CLINICAL TAXONOMY ESTABLISHED
✓ Total Standardized Categories : 25
✓ Tier 1 (Core Motive / NTDs)   : 9 categories (High Clinical Urgency)
✓ Tier 2 (Broad Primary Care)   : 16 categories (Routine Frontline Triage)
✓ Master Label Map Written      : C:\Users\kvcsu_ht23nk8\OneDrive\Desktop\all_Projects\BTech_Project\skin-disease\BTech_Project\Models\metadata\unified_label_map.json


In [3]:
records = []

# =====================================================================
# 1. INGEST CLINICAL MULTISOURCE DATASET (Fitzpatrick17k, ISIC, HAM10000, DermNet)
# =====================================================================
print("=" * 70)
print("1/2 Ingesting Clinical MultiSource Repository...")

df_manifest = pd.read_csv(manifest_path)
clinical_found = 0

for _, row in tqdm(df_manifest.iterrows(), total=len(df_manifest), desc="Indexing MultiSource"):
    raw_rel = row['file_path'].replace('datasets/raw/', '').lstrip('./\\')
    full_path = CLINICAL_DIR / raw_rel
    
    if not full_path.exists():
        continue
        
    clinical_found += 1
    raw_class = str(row['disease_class']).strip()
    unified_class = MANIFEST_MAPPING.get(raw_class, None)
    
    if unified_class is None:
        continue
        
    # Standardize Fitzpatrick scale: only retain genuine phototypes 1 through 6 (-1 means unannotated)
    fz = row.get('fitzpatrick_scale', None)
    try:
        fz_int = int(fz)
        fz_clean = fz_int if 1 <= fz_int <= 6 else None
    except (ValueError, TypeError):
        fz_clean = None

    rel_path_from_root = full_path.relative_to(BTECH_ROOT).as_posix()
    
    records.append({
        "image_id": str(row['image_id']),
        "file_path": rel_path_from_root,
        "unified_class": unified_class,
        "class_id": LABEL_TO_ID[unified_class],
        "clinical_tier": UNIFIED_TAXONOMY[unified_class]["tier"],
        "triage_urgency": UNIFIED_TAXONOMY[unified_class]["urgency"],
        "fitzpatrick_scale": fz_clean,
        "source_dataset": str(row.get('source_dataset', 'clinical_multisource')),
        "original_split": str(row.get('split', 'train'))
    })

print(f"\u2713 Clinical MultiSource Ingested: {clinical_found:,} valid images indexed")

# =====================================================================
# 2. INGEST ORIGINAL DERMNET DATASET (Authoritative Diagnostic Categories)
# =====================================================================
print("\n2/2 Ingesting Original DERMNet Repository...")
dermnet_found = 0

for split in ["train", "test"]:
    split_dir = DERMNET_DIR / split
    if not split_dir.exists():
        continue
        
    for category_folder in split_dir.iterdir():
        if not category_folder.is_dir():
            continue
            
        folder_name = category_folder.name
        
        for img_file in category_folder.iterdir():
            if img_file.suffix.lower() not in [".jpg", ".jpeg", ".png"]:
                continue
                
            dermnet_found += 1
            unified_class = disambiguate_dermnet_class(folder_name, img_file.name)
            
            if unified_class is None or unified_class not in LABEL_TO_ID:
                continue
                
            rel_path_from_root = img_file.relative_to(BTECH_ROOT).as_posix()
            img_id = f"dermnet_{img_file.stem}"
            
            records.append({
                "image_id": img_id,
                "file_path": rel_path_from_root,
                "unified_class": unified_class,
                "class_id": LABEL_TO_ID[unified_class],
                "clinical_tier": UNIFIED_TAXONOMY[unified_class]["tier"],
                "triage_urgency": UNIFIED_TAXONOMY[unified_class]["urgency"],
                "fitzpatrick_scale": None,  # Will be proxy-computed via ITA in EDA phase
                "source_dataset": "dermnet_original",
                "original_split": split
            })

print(f"\u2713 DERMNet Original Ingested   : {dermnet_found:,} valid images indexed")

# =====================================================================
# 3. BUILD MASTER RAW POOL DATAFRAME
# =====================================================================
df_raw_pool = pd.DataFrame(records)

print("=" * 70)
print(f"\u2713 TOTAL UNIFIED IMAGE POOL INDEXED : {len(df_raw_pool):,} images")
print(f"\u2713 Tier 1 (Core Focus / NTDs)       : {(df_raw_pool['clinical_tier'] == 'Tier 1').sum():,} images")
print(f"\u2713 Tier 2 (Routine Primary Care)    : {(df_raw_pool['clinical_tier'] == 'Tier 2').sum():,} images")
print(f"\u2713 Verified Fitzpatrick Annotated   : {df_raw_pool['fitzpatrick_scale'].notna().sum():,} images")
print("=" * 70)

1/2 Ingesting Clinical MultiSource Repository...


Indexing MultiSource: 100%|██████████| 15830/15830 [00:01<00:00, 13277.77it/s]


✓ Clinical MultiSource Ingested: 15,830 valid images indexed

2/2 Ingesting Original DERMNet Repository...
✓ DERMNet Original Ingested   : 19,559 valid images indexed
✓ TOTAL UNIFIED IMAGE POOL INDEXED : 35,389 images
✓ Tier 1 (Core Focus / NTDs)       : 8,710 images
✓ Tier 2 (Routine Primary Care)    : 26,679 images
✓ Verified Fitzpatrick Annotated   : 520 images


In [4]:
# =====================================================================
# 1. STAGE 1: ULTRA-FAST BYTE-SIZE EXTRACTION
# =====================================================================
print("=" * 70)
print("1/3 Stage 1: Extracting File Sizes for Fast Candidate Grouping...")

file_sizes = []
for rel_path in tqdm(df_raw_pool['file_path'].values, desc="File Size Statting"):
    full_p = BTECH_ROOT / rel_path
    try:
        file_sizes.append(full_p.stat().st_size)
    except Exception:
        file_sizes.append(-1)

df_raw_pool['file_size'] = file_sizes

# Filter out unreadable or 0-byte corrupt files
df_valid = df_raw_pool[df_raw_pool['file_size'] > 0].copy()
print(f"\u2713 Valid Readable Files: {len(df_valid):,} / {len(df_raw_pool):,}")

# =====================================================================
# 2. STAGE 2: TARGETED MD5 HASHING FOR SIZE COLLISIONS
# =====================================================================
print("\n2/3 Stage 2: Resolving Size Collisions via Cryptographic MD5...")

size_counts = df_valid['file_size'].value_counts()
potential_dups = set(size_counts[size_counts > 1].index)

hashes = []
collision_count = 0

for idx, row in tqdm(df_valid.iterrows(), total=len(df_valid), desc="Hashing Collisions"):
    sz = row['file_size']
    if sz in potential_dups:
        collision_count += 1
        full_p = BTECH_ROOT / row['file_path']
        try:
            with open(full_p, 'rb') as fp:
                h = hashlib.md5(fp.read()).hexdigest()
            hashes.append(h)
        except Exception:
            hashes.append(f"err_{idx}")
    else:
        # Unique size is guaranteed unique
        hashes.append(f"unique_sz_{sz}")

df_valid['file_hash'] = hashes
print(f"\u2713 Collisions Inspected & Verified: {collision_count:,} candidate files")

# =====================================================================
# 3. CLINICAL CONFLICT RESOLUTION: AUTHORITATIVE GROUND-TRUTH PRIORITY
# =====================================================================
print("\n3/3 Performing Intelligent Deduplication...")

# Deduplication priority hierarchy for identical images:
# 1. Preserves verified clinical Fitzpatrick phototypes (1-6)
# 2. Prioritizes specialized clinical oncology repositories (HAM10000, ISIC, Fitzpatrick17k)
# 3. Prioritizes authoritative original DERMNet labels (protects Tinea & Rosacea from coarse lumping)
def get_source_rank(src):
    if src in ['fitzpatrick17k', 'ham10000', 'isic']:
        return 3
    elif src == 'dermnet_original':
        return 2
    else:
        return 1

df_valid['fitz_priority'] = df_valid['fitzpatrick_scale'].notna().astype(int)
df_valid['source_priority'] = df_valid['source_dataset'].apply(get_source_rank)

# Sort so highest quality annotations come first
df_valid = df_valid.sort_values(
    by=['fitz_priority', 'source_priority'], 
    ascending=[False, False]
)

initial_count = len(df_valid)
df_dedup = df_valid.drop_duplicates(subset=['file_hash'], keep='first').copy()
dropped_count = initial_count - len(df_dedup)

# Clean up helper columns
df_dedup = df_dedup.drop(columns=['file_size', 'file_hash', 'fitz_priority', 'source_priority'])

print("=" * 70)
print("  DEDUPLICATION AUDIT SUMMARY")
print("=" * 70)
print(f"\u2713 Initial Ingested Pool       : {initial_count:,} images")
print(f"\u2713 Exact Duplicates Purged     : {dropped_count:,} redundant images removed")
print(f"\u2713 Clean Unique Master Pool    : {len(df_dedup):,} distinct images retained")
print(f"\u2713 Verified Fitzpatrick Labels : {df_dedup['fitzpatrick_scale'].notna().sum():,} images")
print("=" * 70)

1/3 Stage 1: Extracting File Sizes for Fast Candidate Grouping...


File Size Statting: 100%|██████████| 35389/35389 [00:00<00:00, 41337.99it/s]


✓ Valid Readable Files: 35,389 / 35,389

2/3 Stage 2: Resolving Size Collisions via Cryptographic MD5...


Hashing Collisions: 100%|██████████| 35389/35389 [00:05<00:00, 6262.88it/s] 

✓ Collisions Inspected & Verified: 24,622 candidate files

3/3 Performing Intelligent Deduplication...
  DEDUPLICATION AUDIT SUMMARY
✓ Initial Ingested Pool       : 35,389 images
✓ Exact Duplicates Purged     : 12,398 redundant images removed
✓ Clean Unique Master Pool    : 22,991 distinct images retained
✓ Verified Fitzpatrick Labels : 520 images


In [7]:
# =====================================================================
# 1. COMPILE CLINICAL CLASS DISTRIBUTION BREAKDOWN
# =====================================================================
print("=" * 80)
print("  COMPILING MASTER CLINICAL DISTRIBUTION BREAKDOWN")
print("=" * 80)

summary_rows = []
total_samples = len(df_dedup)

for c_name in CLASS_NAMES:
    c_df = df_dedup[df_dedup['unified_class'] == c_name]
    count = len(c_df)
    tier = UNIFIED_TAXONOMY[c_name]["tier"]
    urgency = UNIFIED_TAXONOMY[c_name]["urgency"]
    fitz_count = c_df['fitzpatrick_scale'].notna().sum()
    pct = (count / total_samples) * 100 if total_samples > 0 else 0
    
    summary_rows.append({
        "Class ID": LABEL_TO_ID[c_name],
        "Unified Disease Class": c_name,
        "Clinical Tier": tier,
        "Triage Urgency": urgency,
        "Total Images": count,
        "Percentage": f"{pct:.2f}%",
        "Fitzpatrick Annotated": fitz_count
    })

df_summary = pd.DataFrame(summary_rows)
# Sort by count descending for clean viewing
df_summary = df_summary.sort_values(by="Total Images", ascending=False)

# Display table
print(df_summary.to_string(index=False))
print("=" * 80)

# =====================================================================
# 2. SAVE MASTER HARMONIZED MANIFESTS & AUDIT LOGS
# =====================================================================
# Save master manifest to metadata
master_manifest_path = METADATA_DIR / "unified_master_manifest.csv"
df_dedup.to_csv(master_manifest_path, index=False)
print(f"\u2713 Master Manifest Saved to      : {master_manifest_path}")

# Save duplicate backup to datasets/metadata
dataset_manifest_path = DATASET_META_DIR / "unified_master_manifest.csv"
df_dedup.to_csv(dataset_manifest_path, index=False)
print(f"\u2713 Backup Manifest Saved to      : {dataset_manifest_path}")

# Save class summary table
summary_csv_path = METADATA_DIR / "class_distribution_summary.csv"
df_summary.to_csv(summary_csv_path, index=False)
print(f"\u2713 Class Summary Table Saved to  : {summary_csv_path}")

# =====================================================================
# 3. HIGH-LEVEL TIER SUMMARY STATS
# =====================================================================
tier1_total = (df_dedup['clinical_tier'] == 'Tier 1').sum()
tier2_total = (df_dedup['clinical_tier'] == 'Tier 2').sum()
fitz_total = df_dedup['fitzpatrick_scale'].notna().sum()

print("\n" + "=" * 80)
print("  PHASE 1 COMPLETE: DATASET HARMONIZATION MILESTONE ACHIEVED")
print("=" * 80)
print(f"\u2713 Final Unique Dataset Size      : {len(df_dedup):,} clinical images")
print(f"\u2713 Tier 1 (Core Focus / NTDs)     : {tier1_total:,} images ({tier1_total / len(df_dedup) * 100:.1f}%)")
print(f"\u2713 Tier 2 (Routine Primary Care)  : {tier2_total:,} images ({tier2_total / len(df_dedup) * 100:.1f}%)")
print(f"\u2713 Verified Fitzpatrick Images   : {fitz_total:,} images ({fitz_total / len(df_dedup) * 100:.1f}%)")
print(f"\u2713 Standardized Target Categories : {len(CLASS_NAMES)} disease classes")
print("=" * 80)


  COMPILING MASTER CLINICAL DISTRIBUTION BREAKDOWN
 Class ID           Unified Disease Class Clinical Tier               Triage Urgency  Total Images Percentage  Fitzpatrick Annotated
       12              Malignant_Melanoma        Tier 1       High-Priority Oncology          2818     12.26%                     79
       18            Seborrheic_Keratosis        Tier 2                 Routine Care          2784     12.11%                      0
        2    Atopic_Dermatitis_and_Eczema        Tier 2           Routine Outpatient          2176      9.46%                     51
       15     Psoriasis_and_Lichen_Planus        Tier 2           Routine Outpatient          1694      7.37%                     98
       13                Melanocytic_Nevi        Tier 2            Routine Screening          1630      7.09%                      2
       21     Tinea_and_Fungal_Infections        Tier 2           Routine Outpatient          1554      6.76%                      0
       14  Onychom